In [ ]:
# """
# Active learning sentiment labelling for MBG tweets.

# Input:
#   - mbg2.csv
#   - mbg2_sample_100_labeled.csv

# Output:
#   - outputs/mbg2_train_1000_indobert_active_learning.csv
#   - outputs/active_learning_candidates_900.csv
#   - outputs/indobert_mbg_sentiment/

# Run:
#   python labelling.py

# Notes:
#   The first 100 labels are treated as human-reviewed baseline labels.
#   The additional samples are selected from the model's least confident
#   predictions, which is the usual active learning pool for manual review.
#   They are still filled with IndoBERT predictions so the file can be used as a
#   weakly labelled training set while you review/correct it.
# """

# from __future__ import annotations

# import argparse
# import inspect
# import json
# import os
# import random
# import re
# from dataclasses import dataclass
# from pathlib import Path
# from typing import Dict, Iterable, Tuple

# import numpy as np
# import pandas as pd
# import torch
# from sklearn.metrics import classification_report
# from sklearn.model_selection import train_test_split
# from torch.utils.data import Dataset
# from transformers import (
#     AutoModelForSequenceClassification,
#     AutoTokenizer,
#     DataCollatorWithPadding,
#     Trainer,
#     TrainingArguments,
#     set_seed,
# )

# os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

# LABELS = ["negatif", "netral", "positif"]
# LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
# ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}


# @dataclass
# class Config:
#     source_csv: Path = Path("mbg2.csv")
#     baseline_csv: Path = Path("mbg2_sample_100_labeled.csv")
#     output_dir: Path = Path("outputs")
#     model_name: str = "indobenchmark/indobert-base-p1"
#     target_size: int = 1000
#     max_length: int = 160
#     seed: int = 42
#     epochs: float = 5.0
#     batch_size: int = 8
#     learning_rate: float = 2e-5
#     warmup_ratio: float = 0.1
#     min_relevance: bool = True


# def clean_text(text: str) -> str:
#     text = str(text)
#     text = re.sub(r"https?://\S+", " URL ", text)
#     text = re.sub(r"@\w+", " USER ", text)
#     text = re.sub(r"#", " ", text)
#     text = re.sub(r"\s+", " ", text).strip()
#     return text


# def is_relevant_mbg(text: str) -> bool:
#     normalized = str(text).lower()
#     return bool(re.search(r"\bmbg\b", normalized)) or "makan bergizi gratis" in normalized


# class TextDataset(Dataset):
#     def __init__(
#         self,
#         texts: Iterable[str],
#         tokenizer: AutoTokenizer,
#         labels: Iterable[int] | None = None,
#         max_length: int = 160,
#     ) -> None:
#         self.texts = list(texts)
#         self.labels = list(labels) if labels is not None else None
#         self.tokenizer = tokenizer
#         self.max_length = max_length

#     def __len__(self) -> int:
#         return len(self.texts)

#     def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
#         item = self.tokenizer(
#             self.texts[index],
#             truncation=True,
#             max_length=self.max_length,
#         )
#         if self.labels is not None:
#             item["labels"] = self.labels[index]
#         return item


# def load_data(config: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
#     source = pd.read_csv(config.source_csv)
#     baseline = pd.read_csv(config.baseline_csv)

#     if "source_row" not in source.columns:
#         source.insert(0, "source_row", source.index + 2)

#     source = source[source["text"].notna()].copy()
#     baseline = baseline[baseline["text"].notna()].copy()

#     if config.min_relevance:
#         source = source[source["text"].apply(is_relevant_mbg)].copy()

#     baseline["text_clean"] = baseline["text"].apply(clean_text)
#     source["text_clean"] = source["text"].apply(clean_text)

#     baseline["sentiment"] = baseline["sentiment"].str.lower().str.strip()
#     baseline = baseline[baseline["sentiment"].isin(LABELS)].copy()
#     baseline["label"] = baseline["sentiment"].map(LABEL_TO_ID)
#     baseline["label_source"] = "baseline_manual"
#     baseline["review_required"] = False

#     # Prefer URL/id matching when present, then fall back to exact cleaned text.
#     used_urls = set(baseline["url"].dropna().astype(str))
#     used_texts = set(baseline["text_clean"].astype(str))
#     pool = source[
#         ~source["url"].astype(str).isin(used_urls)
#         & ~source["text_clean"].astype(str).isin(used_texts)
#     ].copy()

#     pool = pool.drop_duplicates(subset=["text_clean"]).reset_index(drop=True)
#     baseline = baseline.drop_duplicates(subset=["text_clean"]).reset_index(drop=True)
#     return source, baseline, pool


# def split_train_eval(baseline: pd.DataFrame, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     label_counts = baseline["sentiment"].value_counts()
#     can_stratify = len(label_counts) == len(LABELS) and label_counts.min() >= 2
#     train_df, eval_df = train_test_split(
#         baseline,
#         test_size=0.2,
#         random_state=seed,
#         stratify=baseline["sentiment"] if can_stratify else None,
#     )
#     return train_df.reset_index(drop=True), eval_df.reset_index(drop=True)


# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=-1)
#     accuracy = (predictions == labels).mean()
#     return {"accuracy": float(accuracy)}


# def train_model(config: Config, baseline: pd.DataFrame):
#     tokenizer = AutoTokenizer.from_pretrained(config.model_name)
#     model = AutoModelForSequenceClassification.from_pretrained(
#         config.model_name,
#         num_labels=len(LABELS),
#         id2label=ID_TO_LABEL,
#         label2id=LABEL_TO_ID,
#     )

#     train_df, eval_df = split_train_eval(baseline, config.seed)
#     train_dataset = TextDataset(
#         train_df["text_clean"],
#         tokenizer,
#         train_df["label"].astype(int),
#         max_length=config.max_length,
#     )
#     eval_dataset = TextDataset(
#         eval_df["text_clean"],
#         tokenizer,
#         eval_df["label"].astype(int),
#         max_length=config.max_length,
#     )

#     model_dir = config.output_dir / "indobert_mbg_sentiment"
#     training_args = {
#         "output_dir": str(model_dir),
#         "overwrite_output_dir": True,
#         "save_strategy": "epoch",
#         "load_best_model_at_end": True,
#         "metric_for_best_model": "accuracy",
#         "greater_is_better": True,
#         "num_train_epochs": config.epochs,
#         "per_device_train_batch_size": config.batch_size,
#         "per_device_eval_batch_size": config.batch_size,
#         "learning_rate": config.learning_rate,
#         "warmup_ratio": config.warmup_ratio,
#         "weight_decay": 0.01,
#         "logging_steps": 10,
#         "save_total_limit": 2,
#         "report_to": "none",
#         "seed": config.seed,
#     }

#     # Transformers changed this keyword across versions.
#     if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
#         training_args["eval_strategy"] = "epoch"
#     else:
#         training_args["evaluation_strategy"] = "epoch"

#     args = TrainingArguments(**training_args)

#     trainer = Trainer(
#         model=model,
#         args=args,
#         train_dataset=train_dataset,
#         eval_dataset=eval_dataset,
#         tokenizer=tokenizer,
#         data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
#         compute_metrics=compute_metrics,
#     )

#     trainer.train()
#     metrics = trainer.evaluate()

#     eval_predictions = trainer.predict(eval_dataset)
#     y_true = eval_df["sentiment"].tolist()
#     y_pred = [ID_TO_LABEL[int(i)] for i in np.argmax(eval_predictions.predictions, axis=-1)]
#     report = classification_report(y_true, y_pred, labels=LABELS, zero_division=0)

#     model_dir.mkdir(parents=True, exist_ok=True)
#     trainer.save_model(str(model_dir))
#     tokenizer.save_pretrained(str(model_dir))
#     (config.output_dir / "baseline_eval_report.txt").write_text(
#         f"Metrics:\n{metrics}\n\nClassification report:\n{report}\n",
#         encoding="utf-8",
#     )

#     return trainer, tokenizer


# def softmax(logits: np.ndarray) -> np.ndarray:
#     logits = logits - logits.max(axis=1, keepdims=True)
#     exp = np.exp(logits)
#     return exp / exp.sum(axis=1, keepdims=True)


# def margin_and_entropy(probabilities: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
#     sorted_probs = np.sort(probabilities, axis=1)[:, ::-1]
#     margin = sorted_probs[:, 0] - sorted_probs[:, 1]
#     entropy = -np.sum(probabilities * np.log(probabilities + 1e-12), axis=1)
#     return margin, entropy


# def predict_pool(config: Config, trainer: Trainer, tokenizer: AutoTokenizer, pool: pd.DataFrame) -> pd.DataFrame:
#     dataset = TextDataset(pool["text_clean"], tokenizer, max_length=config.max_length)
#     predictions = trainer.predict(dataset)
#     probabilities = softmax(predictions.predictions)
#     pred_ids = probabilities.argmax(axis=1)
#     confidence = probabilities.max(axis=1)
#     margin, entropy = margin_and_entropy(probabilities)

#     scored = pool.copy()
#     scored["predicted_sentiment"] = [ID_TO_LABEL[int(idx)] for idx in pred_ids]
#     scored["model_confidence"] = confidence
#     scored["uncertainty_margin"] = margin
#     scored["uncertainty_entropy"] = entropy

#     for label, idx in LABEL_TO_ID.items():
#         scored[f"prob_{label}"] = probabilities[:, idx]

#     return scored


# def select_active_learning_batch(config: Config, baseline: pd.DataFrame, scored_pool: pd.DataFrame) -> pd.DataFrame:
#     needed = max(0, config.target_size - len(baseline))
#     if needed == 0:
#         return scored_pool.head(0).copy()

#     selected = (
#         scored_pool.sort_values(
#             by=["uncertainty_margin", "uncertainty_entropy"],
#             ascending=[True, False],
#         )
#         .head(needed)
#         .copy()
#     )

#     selected.insert(0, "sample_id", range(len(baseline) + 1, len(baseline) + len(selected) + 1))
#     selected["sentiment"] = selected["predicted_sentiment"]
#     selected["label_source"] = "indobert_prediction_active_learning"
#     selected["review_required"] = True
#     return selected


# def build_train_1000(config: Config, baseline: pd.DataFrame, selected: pd.DataFrame) -> pd.DataFrame:
#     baseline_out = baseline.copy()
#     baseline_out["predicted_sentiment"] = baseline_out["sentiment"]
#     baseline_out["model_confidence"] = 1.0
#     baseline_out["uncertainty_margin"] = np.nan
#     baseline_out["uncertainty_entropy"] = np.nan
#     for label in LABELS:
#         baseline_out[f"prob_{label}"] = np.where(baseline_out["sentiment"] == label, 1.0, 0.0)

#     common_columns = [
#         "sample_id",
#         "source_row",
#         "id",
#         "url",
#         "text",
#         "createdAt",
#         "retweetCount",
#         "replyCount",
#         "likeCount",
#         "quoteCount",
#         "viewCount",
#         "lang",
#         "isReply",
#         "isRetweet",
#         "isQuote",
#         "sentiment",
#         "predicted_sentiment",
#         "model_confidence",
#         "uncertainty_margin",
#         "uncertainty_entropy",
#         "prob_negatif",
#         "prob_netral",
#         "prob_positif",
#         "label_source",
#         "review_required",
#     ]

#     for frame in (baseline_out, selected):
#         for column in common_columns:
#             if column not in frame.columns:
#                 frame[column] = np.nan

#     train_1000 = pd.concat(
#         [baseline_out[common_columns], selected[common_columns]],
#         ignore_index=True,
#     )
#     return train_1000.head(config.target_size)


# def save_outputs(config: Config, baseline: pd.DataFrame, selected: pd.DataFrame, train_1000: pd.DataFrame) -> None:
#     config.output_dir.mkdir(parents=True, exist_ok=True)
#     selected.to_csv(config.output_dir / "active_learning_candidates_900.csv", index=False)
#     train_1000.to_csv(config.output_dir / "mbg2_train_1000_indobert_active_learning.csv", index=False)

#     summary = {
#         "baseline_rows": int(len(baseline)),
#         "selected_rows": int(len(selected)),
#         "train_rows": int(len(train_1000)),
#         "train_label_distribution": train_1000["sentiment"].value_counts().to_dict(),
#         "review_required_rows": int(train_1000["review_required"].sum()),
#     }
#     with (config.output_dir / "active_learning_summary.json").open("w", encoding="utf-8") as file:
#         json.dump(summary, file, ensure_ascii=False, indent=2)


# def parse_args() -> Config:
#     parser = argparse.ArgumentParser(description="Active learning with IndoBERT for MBG sentiment labelling.")
#     parser.add_argument("--source-csv", default="mbg2.csv")
#     parser.add_argument("--baseline-csv", default="mbg2_sample_100_labeled.csv")
#     parser.add_argument("--output-dir", default="outputs")
#     parser.add_argument("--model-name", default="indobenchmark/indobert-base-p1")
#     parser.add_argument("--target-size", type=int, default=1000)
#     parser.add_argument("--max-length", type=int, default=160)
#     parser.add_argument("--seed", type=int, default=42)
#     parser.add_argument("--epochs", type=float, default=5.0)
#     parser.add_argument("--batch-size", type=int, default=8)
#     parser.add_argument("--learning-rate", type=float, default=2e-5)
#     parser.add_argument("--no-mbg-filter", action="store_true")
#     args = parser.parse_args()

#     return Config(
#         source_csv=Path(args.source_csv),
#         baseline_csv=Path(args.baseline_csv),
#         output_dir=Path(args.output_dir),
#         model_name=args.model_name,
#         target_size=args.target_size,
#         max_length=args.max_length,
#         seed=args.seed,
#         epochs=args.epochs,
#         batch_size=args.batch_size,
#         learning_rate=args.learning_rate,
#         min_relevance=not args.no_mbg_filter,
#     )


# def main() -> None:
#     config = parse_args()
#     random.seed(config.seed)
#     np.random.seed(config.seed)
#     torch.manual_seed(config.seed)
#     set_seed(config.seed)

#     config.output_dir.mkdir(parents=True, exist_ok=True)

#     source, baseline, pool = load_data(config)
#     if len(baseline) >= config.target_size:
#         raise ValueError("Baseline sudah lebih besar atau sama dengan target_size.")
#     if len(pool) < config.target_size - len(baseline):
#         raise ValueError(
#             f"Pool hanya {len(pool)} baris, tidak cukup untuk target {config.target_size}."
#         )

#     print(f"Source relevant rows: {len(source)}")
#     print(f"Baseline rows: {len(baseline)}")
#     print(f"Unlabelled pool rows: {len(pool)}")
#     print("Baseline distribution:")
#     print(baseline["sentiment"].value_counts())

#     trainer, tokenizer = train_model(config, baseline)
#     scored_pool = predict_pool(config, trainer, tokenizer, pool)
#     selected = select_active_learning_batch(config, baseline, scored_pool)
#     train_1000 = build_train_1000(config, baseline, selected)
#     save_outputs(config, baseline, selected, train_1000)

#     print("\nSaved:")
#     print(config.output_dir / "mbg2_train_1000_indobert_active_learning.csv")
#     print(config.output_dir / "active_learning_candidates_900.csv")
#     print(config.output_dir / "baseline_eval_report.txt")
#     print(config.output_dir / "active_learning_summary.json")
#     print("\nFinal distribution:")
#     print(train_1000["sentiment"].value_counts())


# if __name__ == "__main__":
#     main()